# NYXVAULT — этап 2: обучение следованию инструкциям (SFT)

Этот ноутбук является частью того же GitHub-репозитория NYXVAULT. Он использует базовую модель из `/content/nyxvault-output/checkpoints/step_600.pt`, создаёт 5 000 instruction-примеров и запускает отдельный SFT-этап на 100 шагов.

**Важно:** базовый checkpoint `step_600.pt` и токенизатор не перезаписываются. SFT-файлы сохраняются отдельно в `/content/nyxvault-output/sft_checkpoints`.

Сначала завершите базовое обучение и сохраните `nyxvault-output` на Google Drive. В Colab выберите GPU.

In [ ]:
import os
import sys
import json
import shutil
import zipfile
import urllib.request
import subprocess
from pathlib import Path

ROOT = Path("/content")
WORK = ROOT / "nyxvault"
OUT = ROOT / "nyxvault-output"

# Проверяем существующие веса: этот ноутбук не запускает базовое обучение заново.
BASE = OUT / "checkpoints" / "step_600.pt"
TOKENIZER = OUT / "tokenizer.json"

if not BASE.is_file():
    raise FileNotFoundError(f"Не найден базовый checkpoint: {BASE}. Восстановите nyxvault-output из Google Drive.")
if not TOKENIZER.is_file():
    raise FileNotFoundError(f"Не найден токенизатор: {TOKENIZER}. Восстановите nyxvault-output из Google Drive.")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU не подключена. В Colab откройте Runtime → Change runtime type → T4 GPU.")

print("GPU:", torch.cuda.get_device_name(0))
base = torch.load(BASE, map_location="cpu", weights_only=False)
print("Базовая версия:", int(base["step"]) + 1, "шагов")
print("Размер словаря:", int(base["vocab_size"]))
if "optimizer" not in base:
    raise RuntimeError("step_600.pt не содержит состояния оптимизатора. Нужен полный step_600.pt checkpoint.")

# Каждый запуск получает актуальный main из GitHub ZIP; папку результатов не трогаем.
if WORK.exists():
    shutil.rmtree(WORK)
archive = ROOT / "nyxvault-main.zip"
url = "https://codeload.github.com/Phantom220-hub/Nyxvault/zip/refs/heads/main"
request = urllib.request.Request(url, headers={"User-Agent": "NYXVAULT-Colab"})
with urllib.request.urlopen(request, timeout=120) as response, archive.open("wb") as target:
    shutil.copyfileobj(response, target)

WORK.mkdir(parents=True, exist_ok=True)
prefix = "Nyxvault-main/"
with zipfile.ZipFile(archive) as zf:
    for member in zf.infolist():
        if not member.filename.startswith(prefix):
            continue
        relative = Path(member.filename[len(prefix):])
        if not relative.parts:
            continue
        target = (WORK / relative).resolve()
        if WORK.resolve() not in target.parents and target != WORK.resolve():
            raise RuntimeError("Недопустимый путь в GitHub ZIP.")
        if member.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(member) as src, target.open("wb") as dst:
                shutil.copyfileobj(src, dst)
archive.unlink(missing_ok=True)

assert (WORK / "training/finetune_instructions.py").is_file()
assert (WORK / "training/generate_instruction_dataset.py").is_file()
print("Актуальный код NYXVAULT загружен из GitHub main.")

## 1. Создать датасет и проверить его

Генератор создаёт 5 000 строк: Python, JavaScript/TypeScript, алгоритмы, отладка, Godot/GDScript. Это стартовый корпус с авторскими шаблонами и вариантами параметров, а не 5 000 независимо проверенных человеком решений.

In [ ]:
# Генерация 5 000 записей
subprocess.run(
    [sys.executable, "training/generate_instruction_dataset.py",
     "--out", "training/data/instruction_sft.jsonl"],
    cwd=WORK, check=True
)

# Проверка схемы, тем, синтаксиса Python/JSON, уникальных ID и дубликатов.
REPORT = OUT / "instruction_sft_validation.json"
subprocess.run(
    [sys.executable, "training/validate_data.py",
     "training/data/instruction_sft.jsonl",
     "--report-json", str(REPORT)],
    cwd=WORK, check=True
)

# Дополнительно исполняем доверенные Python-примеры одной сессией,
# чтобы не запускать тысячи отдельных процессов из валидатора.
import re
dataset_path = WORK / "training/data/instruction_sft.jsonl"
rows = [json.loads(line) for line in dataset_path.read_text(encoding="utf-8").splitlines() if line.strip()]
assert len(rows) == 5000, f"Ожидалось 5000 примеров, получено {len(rows)}"

from collections import Counter
print("Записей по категориям:", dict(Counter(row["tags"][0] for row in rows)))

passed = 0
for row in rows:
    if row["language"] != "python" or not row.get("tests"):
        continue
    match = re.search(r"\x60{3}python\s*(.*?)\x60{3}", row["output"], flags=re.S)
    if not match:
        raise RuntimeError(f"Не найден Python-блок в {row['id']}")
    namespace = {}
    exec(compile(match.group(1), row["id"], "exec"), namespace)
    for test in row["tests"]:
        exec(compile(test, row["id"] + "_test", "exec"), namespace)
    passed += 1

print("Python test cases executed:", passed)
print("Отчёт валидатора:", REPORT)
if passed < 2500:
    raise RuntimeError("Слишком мало исполняемых Python-примеров: проверьте генератор.")

## 2. Первый SFT-этап — 100 шагов

Для первого запуска берём базовые веса с шага 600. Создаётся новый оптимизатор, потому что цель меняется: теперь модель учится отвечать на инструкции. Результат сохраняется отдельно.

In [ ]:
SFT_OUT = OUT / "sft_checkpoints"
subprocess.run(
    [
        sys.executable, "training/finetune_instructions.py",
        "--data", "training/data/instruction_sft.jsonl",
        "--tokenizer", str(TOKENIZER),
        "--init-from", str(BASE),
        "--steps", "1000",
        "--max-steps", "100",
        "--batch-size", "2",
        "--grad-accum-steps", "4",
        "--block-size", "512",
        "--lr", "5e-5",
        "--warmup-steps", "50",
        "--eval-every", "25",
        "--save-every", "100",
        "--out", str(SFT_OUT),
    ],
    cwd=WORK, check=True
)

SFT_STEP = SFT_OUT / "sft_step_100.pt"
BEST_SFT = SFT_OUT / "best_sft.pt"
if not SFT_STEP.is_file():
    raise RuntimeError("SFT завершился, но sft_step_100.pt не найден.")
print("SFT checkpoint:", SFT_STEP)
print("Best weights exist:", BEST_SFT.is_file())

## 3. Проверить генерацию после SFT

Проверка ниже использует лучший SFT checkpoint, если он создан. Ответ нужно оценивать запуском тестов, а не только по тому, насколько убедительно выглядит текст.

In [ ]:
if BEST_SFT.is_file():
    subprocess.run(
        [
            sys.executable, "training/generate_instruction.py",
            "--checkpoint", str(BEST_SFT),
            "--tokenizer", str(TOKENIZER),
            "--task", "Создай функцию Python inventory_count(items, item_id), которая считает предметы по ID. Добавь обработку пустого списка и пример использования.",
            "--language", "python",
            "--topic", "simple-classes",
            "--level", "2",
            "--type", "write_code",
            "--max-new", "180",
            "--temperature", "0",
        ],
        cwd=WORK, check=True
    )
else:
    print("best_sft.pt пока нет: проверьте validation loss и сообщения обучения.")

## 4. Продолжить SFT позже

После сохранения папки `nyxvault-output` на Google Drive для следующего этапа используйте только последний полный checkpoint `sft_step_100.pt`. Не используйте `best_sft.pt` для resume: он хранит только лучшие веса, без состояния оптимизатора.

In [ ]:
# Запускайте эту ячейку отдельно только для следующего этапа (101–200).
# Она не выполняется автоматически при первом SFT-запуске.
#
# subprocess.run([
#     sys.executable, "training/finetune_instructions.py",
#     "--data", "training/data/instruction_sft.jsonl",
#     "--tokenizer", str(TOKENIZER),
#     "--resume", str(SFT_OUT / "sft_step_100.pt"),
#     "--steps", "1000", "--max-steps", "100",
#     "--batch-size", "2", "--grad-accum-steps", "4",
#     "--block-size", "512", "--lr", "5e-5", "--warmup-steps", "50",
#     "--eval-every", "25", "--save-every", "100",
#     "--out", str(SFT_OUT)
# ], cwd=WORK, check=True)

## 5. Сохранение

Папка `/content` временная. Сохраните `/content/nyxvault-output` на Google Drive после каждого этапа. Исходный базовый checkpoint `checkpoints/step_600.pt` не меняется; SFT checkpoint лежит в `sft_checkpoints/`.